# AG2 Multi-Agent RAG with Weaviate

<a href="https://colab.research.google.com/github/weaviate/weaviate-examples/blob/main/ag2-multiagent-rag/ag2_multiagent_rag_with_weaviate.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook demonstrates how to use [AG2](https://ag2.ai/) (formerly AutoGen)
multi-agent conversations with [Weaviate](https://weaviate.io/) as the vector
database for Retrieval-Augmented Generation (RAG).

**AG2** is a multi-agent conversation framework with 500K+ monthly PyPI downloads,
4,300+ GitHub stars, and 400+ contributors.

**Weaviate** is an open-source vector database that stores both objects and vectors,
enabling vector search combined with structured filtering.

In this example, two AG2 agents collaborate:
- **Research Agent**: Retrieves relevant documents from Weaviate via semantic search
- **Analyst Agent**: Synthesizes retrieved information into comprehensive answers

## Install Dependencies

In [ ]:
!pip install -U "ag2[openai]>=0.11.4,<1.0" weaviate-client

## Imports and Configuration

In [ ]:
import os

from autogen import (
    AssistantAgent,
    GroupChat,
    GroupChatManager,
    LLMConfig,
    UserProxyAgent,
)
import weaviate
from weaviate.classes.config import Configure, DataType, Property
from weaviate.classes.query import MetadataQuery

# Set your OpenAI API key (used for both AG2 and Weaviate text2vec-openai)
os.environ["OPENAI_API_KEY"] = "your-api-key"  # Replace with your key

## Connect to Weaviate

We'll use **Weaviate Embedded** which runs in-process — no Docker or server needed.
For production, use [Weaviate Cloud](https://console.weaviate.cloud/) or
a self-hosted Docker deployment.

In [ ]:
# Connect to Weaviate Embedded (in-process, no Docker needed)
client = weaviate.connect_to_embedded(
    headers={"X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"]},
)

print(f"Weaviate is ready: {client.is_ready()}")

collection_name = "AG2_RAG_Demo"

# Delete collection if exists (for re-running)
if client.collections.exists(collection_name):
    client.collections.delete(collection_name)

# Create collection with OpenAI vectorizer
collection = client.collections.create(
    name=collection_name,
    vector_config=Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    properties=[
        Property(name="text", data_type=DataType.TEXT),
        Property(name="source", data_type=DataType.TEXT),
    ],
)

print(f"Collection '{collection_name}' created.")

## Prepare and Index Sample Data

We'll index a small set of documents about AI concepts. Weaviate automatically
generates embeddings using the configured `text2vec-openai` vectorizer — no manual
embedding step required.

In [ ]:
# Sample documents about AI topics
documents = [
    {
        "text": (
            "Retrieval-Augmented Generation (RAG) is a technique that combines "
            "information retrieval with language model generation. It first retrieves "
            "relevant documents from a knowledge base, then uses them as context for "
            "generating accurate, grounded responses. RAG reduces hallucination and "
            "enables models to access up-to-date information beyond their training data."
        ),
        "source": "ai_concepts.md",
    },
    {
        "text": (
            "Vector databases store data as high-dimensional vectors (embeddings) "
            "and enable fast similarity search. Weaviate is an open-source vector database "
            "that combines vector search with structured filtering and supports multiple "
            "vectorizer modules including OpenAI, Cohere, and Hugging Face transformers."
        ),
        "source": "vector_databases.md",
    },
    {
        "text": (
            "Multi-agent systems use multiple AI agents that collaborate to solve "
            "complex tasks. Each agent can have specialized roles, tools, and knowledge. "
            "AG2 (formerly AutoGen) is a popular framework for building multi-agent "
            "conversations where agents can use tools, write code, and coordinate "
            "through structured dialogue patterns."
        ),
        "source": "multi_agent_systems.md",
    },
    {
        "text": (
            "Embedding models convert text into dense numerical vectors that "
            "capture semantic meaning. Similar texts produce similar vectors, enabling "
            "semantic search. Popular embedding models include OpenAI text-embedding-3-small, "
            "sentence-transformers, and BGE models. The choice of embedding model "
            "significantly impacts retrieval quality in RAG systems."
        ),
        "source": "embeddings.md",
    },
    {
        "text": (
            "Chunking is the process of splitting documents into smaller pieces "
            "for embedding and retrieval. Common strategies include fixed-size chunking, "
            "recursive character splitting, and semantic chunking. Optimal chunk size "
            "depends on the use case: 256-512 tokens for precise retrieval, 1000+ tokens "
            "for broader context. Overlap between chunks helps preserve context."
        ),
        "source": "chunking_strategies.md",
    },
]

# Insert documents — Weaviate auto-vectorizes via text2vec-openai
collection = client.collections.get(collection_name)

with collection.batch.dynamic() as batch:
    for doc in documents:
        batch.add_object(properties=doc)

print(
    f"Indexed {len(documents)} documents into Weaviate collection '{collection_name}'"
)

## Test Semantic Search

Verify that Weaviate retrieval works before connecting it to AG2 agents.

In [ ]:
# Quick test — Weaviate near_text handles embedding automatically
results = collection.query.near_text(
    query="What is RAG?",
    limit=3,
    return_metadata=MetadataQuery(distance=True),
)

for obj in results.objects:
    print(f"Source: {obj.properties['source']} (distance: {obj.metadata.distance:.4f})")
    print(f"  {obj.properties['text'][:100]}...")
    print()

## Define Weaviate Search Function for AG2

Create a search function that AG2 agents will use as a registered tool to retrieve
relevant documents from the Weaviate collection.

In [ ]:
def search_weaviate(query: str, top_k: int = 3) -> str:
    """
    Search Weaviate collection for relevant documents.

    Args:
        query: Search query string.
        top_k: Number of results to return.

    Returns:
        Formatted string with retrieved documents and sources.
    """
    results = collection.query.near_text(
        query=query,
        limit=top_k,
        return_metadata=MetadataQuery(distance=True),
    )

    formatted = []
    for i, obj in enumerate(results.objects, 1):
        source = obj.properties["source"]
        text = obj.properties["text"]
        distance = obj.metadata.distance
        formatted.append(f"[{i}] Source: {source} (distance: {distance:.4f})\n{text}")

    return (
        "\n\n---\n\n".join(formatted) if formatted else "No relevant documents found."
    )

## Set Up AG2 Multi-Agent System

Create AG2 agents that collaborate to answer questions using Weaviate retrieval:

1. **Research Agent** — Uses the `search_weaviate` tool to find relevant documents
2. **Analyst Agent** — Synthesizes retrieved information into a final answer
3. **User Proxy** — Orchestrates the conversation and executes tool calls

In [ ]:
# AG2 LLM Configuration
llm_config = LLMConfig(
    {
        "model": "gpt-4o-mini",
        "api_key": os.environ["OPENAI_API_KEY"],
        "api_type": "openai",
    }
)

# Create agents
researcher = AssistantAgent(
    name="researcher",
    system_message=(
        "You are a research agent. When asked a question, use the search_documents "
        "tool to retrieve relevant documents from the Weaviate knowledge base. "
        "Present your findings clearly with source references. If results are "
        "insufficient, try rephrasing your search query."
    ),
    llm_config=llm_config,
)

analyst = AssistantAgent(
    name="analyst",
    system_message=(
        "You are an analyst. Based on the researcher's findings, synthesize the "
        "information into a comprehensive, well-structured answer. Always reference "
        "the source documents. End with TERMINATE when done."
    ),
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config=False,
    is_termination_msg=lambda x: x.get("content", "")
    and "TERMINATE" in x.get("content", ""),
)


# Register Weaviate search as AG2 tool
@user_proxy.register_for_execution()
@researcher.register_for_llm(
    description=(
        "Search the Weaviate vector database for relevant documents. "
        "Returns document chunks with similarity distances and source references. "
        "Use specific search queries for best results."
    )
)
def search_documents(query: str, top_k: int = 3) -> str:
    """Search Weaviate for relevant documents."""
    return search_weaviate(query, top_k)


# Set up GroupChat
group_chat = GroupChat(
    agents=[user_proxy, researcher, analyst],
    messages=[],
    max_round=12,
)

manager = GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
)

# Run the multi-agent conversation
user_proxy.run(
    manager,
    message=(
        "What is RAG and how does it work with vector databases? "
        "Also explain how multi-agent systems can improve RAG quality."
    ),
).process()

## Cleanup

Close the Weaviate client connection and clean up resources.

In [ ]:
# Delete the demo collection
client.collections.delete(collection_name)
print(f"Deleted collection '{collection_name}'")

# Close the Weaviate client
client.close()
print("Weaviate connection closed.")

## Summary

This notebook demonstrated:

1. **Weaviate** as a vector database with automatic embedding via `text2vec-openai`
2. **AG2** multi-agent conversations with tool-calling capabilities
3. **Combining both**: AG2 agents using Weaviate semantic search as a registered tool
   for grounded, citation-backed RAG responses

### Key Components

| Component | Role |
|-----------|------|
| **Weaviate Embedded** | In-process vector database (no Docker needed) |
| **text2vec-openai** | Automatic embedding via OpenAI API |
| **AG2 UserProxy** | Orchestrates conversation, executes tools |
| **AG2 Research Agent** | Calls `search_documents` tool |
| **AG2 Analyst Agent** | Synthesizes findings into final answer |

### Next Steps

- Scale up with more documents and a Weaviate Cloud deployment
- Add more specialized agents (fact-checker, summarizer, etc.)
- Use Weaviate hybrid search (vector + BM25) for better retrieval
- Add metadata filtering for domain-specific queries
- Explore Weaviate generative modules for built-in RAG

### Resources

- [AG2 Documentation](https://docs.ag2.ai/)
- [Weaviate Documentation](https://weaviate.io/developers/weaviate)
- [AG2 GitHub](https://github.com/ag2ai/ag2)
- [Weaviate GitHub](https://github.com/weaviate/weaviate)